In [1]:
import sqlite3
import pandas as pd


conn = sqlite3.connect("churn.db")
cursor = conn.cursor()

# Step 8 — rule-based baseline risk score
cursor.executescript("""
DROP VIEW IF EXISTS risk_score_baseline;

CREATE VIEW risk_score_baseline AS
SELECT
    customerID,
    Contract,
    tenure,
    MonthlyCharges,
    TechSupport,
    InternetService,
    Churn,
    (
        CASE WHEN Contract = 'Month-to-month' THEN 3 ELSE 0 END +
        CASE WHEN tenure <= 12 THEN 2 ELSE 0 END +
        CASE WHEN TechSupport = 'No' THEN 1 ELSE 0 END +
        CASE WHEN InternetService = 'Fiber optic' THEN 1 ELSE 0 END +
        CASE WHEN MonthlyCharges >= 70 THEN 1 ELSE 0 END
    ) AS rule_risk_score
FROM customers;
""")

# Step 10 — master view joining raw customers + rule_risk_score
cursor.executescript("""
DROP VIEW IF EXISTS customer_master;

CREATE VIEW customer_master AS
SELECT
    c.*,
    r.rule_risk_score
FROM customers c
JOIN risk_score_baseline r ON c.customerID = r.customerID;
""")

conn.commit()
print("Views created successfully.")

# Now confirm they exist
print(pd.read_sql("SELECT name, type FROM sqlite_master WHERE type='view';", conn))

Views created successfully.
                  name  type
0  risk_score_baseline  view
1      customer_master  view


In [2]:
df = pd.read_sql("SELECT * FROM customer_master;", conn)
print(df.shape)
df.head()

(7043, 22)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,rule_risk_score
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,6
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,1
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,6
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,8


In [3]:
# TotalCharges sometimes has blank strings for brand-new customers (tenure=0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)

# Check for any other missing values
print(df.isnull().sum().sum(), "missing values total")

# Target variable: 1 = churned, 0 = stayed
y = (df["Churn"] == "Yes").astype(int)
print(y.value_counts())

0 missing values total
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
from sklearn.preprocessing import LabelEncoder

# Drop columns that aren't real predictive features
X = df.drop(columns=["customerID", "Churn", "rule_risk_score"])

# Encode every text/categorical column
encoders = {}
for col in X.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

print(X.dtypes)
X.head()

gender                int32
SeniorCitizen         int64
Partner               int32
Dependents            int32
tenure                int64
PhoneService          int32
MultipleLines         int32
InternetService       int32
OnlineSecurity        int32
OnlineBackup          int32
DeviceProtection      int32
TechSupport           int32
StreamingTV           int32
StreamingMovies       int32
Contract              int32
PaperlessBilling      int32
PaymentMethod         int32
MonthlyCharges      float64
TotalCharges        float64
dtype: object


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,1,0,1,0,0,2,0,0,0,0,0,1,2,29.85,29.85
1,1,0,0,0,34,1,0,0,2,0,2,0,0,0,1,0,3,56.95,1889.50
2,1,0,0,0,2,1,0,0,2,2,0,0,0,0,0,1,3,53.85,108.15
3,1,0,0,0,45,0,1,0,2,0,2,2,0,0,1,0,0,42.30,1840.75
4,0,0,0,0,2,1,0,1,0,0,0,0,0,0,0,1,2,70.70,151.65


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keeps the 73.5/26.5 churn ratio consistent in both sets
)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

Train size: 5634
Test size : 1409


In [6]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

print("Model trained.")

Model trained.


In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("ROC-AUC :", round(roc_auc_score(y_test, y_proba), 3))
print()
print(classification_report(y_test, y_pred, target_names=["Stayed", "Churned"]))
print("Confusion matrix [[TN, FP],[FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.732
ROC-AUC : 0.838

              precision    recall  f1-score   support

      Stayed       0.91      0.71      0.80      1035
     Churned       0.50      0.80      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.80      0.73      0.75      1409

Confusion matrix [[TN, FP],[FN, TP]]:
[[732 303]
 [ 74 300]]


In [8]:
from sklearn.metrics import roc_auc_score

# Your SQL rule_risk_score, normalized to 0-1 like a probability
baseline_score = df.loc[X_test.index, "rule_risk_score"] / 8   # max possible score was 8

baseline_auc = roc_auc_score(y_test, baseline_score)
ml_auc = roc_auc_score(y_test, y_proba)

print(f"SQL rule-based baseline AUC : {baseline_auc:.3f}")
print(f"ML model (Logistic Reg) AUC : {ml_auc:.3f}")
print(f"Improvement: {ml_auc - baseline_auc:+.3f}")

SQL rule-based baseline AUC : 0.821
ML model (Logistic Reg) AUC : 0.838
Improvement: +0.017


In [9]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", key=abs, ascending=False)

print(coef_df.head(10))

             Feature  Coefficient
14          Contract    -0.825016
15  PaperlessBilling     0.402394
5       PhoneService    -0.388280
8     OnlineSecurity    -0.275501
3         Dependents    -0.254447
11       TechSupport    -0.252949
1      SeniorCitizen     0.227759
9       OnlineBackup    -0.127437
6      MultipleLines     0.101167
16     PaymentMethod     0.078289


In [10]:
all_proba = model.predict_proba(X)[:, 1]

risk_df = df[["customerID", "Contract", "tenure", "MonthlyCharges", "Churn", "rule_risk_score"]].copy()
risk_df["ChurnProbability"] = all_proba.round(3)
risk_df["RiskTier"] = pd.cut(
    risk_df["ChurnProbability"],
    bins=[0, 0.2, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High", "Critical"]
)

risk_df.to_sql("customer_risk_scores", conn, if_exists="replace", index=False)

print(risk_df.sort_values("ChurnProbability", ascending=False).head(10))

      customerID        Contract  tenure  MonthlyCharges Churn  \
3380  5178-LMXOP  Month-to-month       1           95.10   Yes   
4800  9300-AGZNL  Month-to-month       1           94.00   Yes   
1976  9497-QCMMS  Month-to-month       1           93.55   Yes   
6368  2720-WGKHP  Month-to-month       2           94.00   Yes   
3749  4424-TKOPW  Month-to-month       2           93.85   Yes   
3159  5150-ITWWB  Month-to-month       3           94.85    No   
2208  7216-EWTRS  Month-to-month       1          100.80   Yes   
5989  5567-WSELE  Month-to-month       3           94.60   Yes   
301   8098-LLAZX  Month-to-month       4           95.45   Yes   
1410  7024-OHCCK  Month-to-month       2           93.85   Yes   

      rule_risk_score  ChurnProbability  RiskTier  
3380                8             0.936  Critical  
4800                8             0.934  Critical  
1976                8             0.934  Critical  
6368                8             0.933  Critical  
3749         

In [11]:
active_risk.to_sql("retention_action_list", conn, if_exists="replace", index=False)
print("Saved table: retention_action_list")
active_risk.to_sql("retention_action_list", conn, if_exists="replace", index=False)
print("Saved table: retention_action_list")

NameError: name 'active_risk' is not defined